# Does investing ethically cost you returns?
**iShares MSCI World SRI vs iShares Core MSCI World, in GBP, Oct 2017 onwards**

Historical analysis only. Nothing here is investment advice.

## 1. Data: download and clean
The pipeline lives in `src/` so this notebook stays readable. `build()` downloads (or loads cached) prices, FX, SONIA and Fama-French factors, then cleans them. Every cleaning decision is written to `data/processed/data_quality.md`.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))   # so `src` imports from notebooks/

import pandas as pd
import matplotlib.pyplot as plt
from src import config
from src.build_dataset import build

data = build(refresh=False)
daily, monthly = data["daily"], data["monthly"]

### 1a. Sanity checks
Before any analysis, check that the cleaned data looks like the funds really behaved:
- both series start on the same date and have no gaps
- large daily moves happen in **both** funds on the same day (market news), not just one (bad data)
- the growth of £1 roughly matches the factsheet performance figures

In [ ]:
print(daily.index.min(), "to", daily.index.max(), "|", daily.isna().sum().to_dict())
data["large_moves"].round(3)

In [ ]:
growth = (1 + monthly[["SRI", "CONV"]]).cumprod() * 10_000
ax = growth.plot(figsize=(9, 4), title="Growth of £10,000 (GBP, dividends reinvested)")
ax.set_ylabel("£"); plt.show()
growth.iloc[-1].round(0)

### 1b. Cross-check against the factsheets
Compare the calendar-year returns below with the 'discrete annual performance' table on each iShares factsheet. Small gaps are expected (we use market price, the factsheet uses NAV; and the SRI fund's factsheet return is in EUR), but differences of more than ~1 percentage point mean something is wrong.

In [ ]:
calendar = (1 + monthly[["SRI", "CONV"]]).groupby(monthly.index.year).prod() - 1
calendar.style.format("{:.1%}")

## 2. Headline metrics
*(next step)*